# Day 44 — Solutions: Model Deployment with FastAPI
Train + save model (joblib), serve predictions via FastAPI, test with curl.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
clf.score(Xte, yte), joblib.dump(clf, 'model.joblib')

## Minimal FastAPI app (save as app.py)

In [ ]:
app_py = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, conlist
import joblib, numpy as np

app = FastAPI()
model = joblib.load('model.joblib')
CLASS_NAMES = ['setosa', 'versicolor', 'virginica']

class IrisFeatures(BaseModel):
    features: conlist(float, min_items=4, max_items=4)

@app.post('/predict')
def predict(data: IrisFeatures):
    try:
        X = np.array([data.features], dtype=float)
        pred = int(model.predict(X).tolist()[0])
        return {'prediction': pred, 'class_name': CLASS_NAMES[pred]}
    except Exception as e:
        raise HTTPException(status_code=400, detail=f'Bad request: {e}')
'''

open('app.py','w').write(app_py)
'Wrote app.py'

## Run server (in terminal)
uvicorn app:app --reload

## Test with curl
curl -s -X POST http://127.0.0.1:8000/predict -H 'Content-Type: application/json' -d '{"features": [5.1, 3.5, 1.4, 0.2]}'